# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation via AWS Agentcore, and observability trace extraction.

### 0. Install Dependencies & Setup
Ensure the environment has the required libraries and initialize variables.

In [ ]:
!pip install boto3 requests

import boto3
import requests
import json

# Initialize variables to prevent NameError in later cells
access_token = None

### 1. Authenticate with Amazon Cognito
Authentication is performed against the live Cognito User Pool provisioned via Terraform.

In [ ]:
REGION = "us-east-1"
CLIENT_ID = "2r1ik1k110jbu6nfmuoegk5lns"  # Injected from terraform output
USERNAME = "analyst_user"
PASSWORD = "SecurePassword123!"

if CLIENT_ID == "<YOUR_COGNITO_CLIENT_ID>":
    print("❌ ERROR: Please replace <YOUR_COGNITO_CLIENT_ID> with your actual Client ID before running this cell.")
else:
    # Authenticate against Cognito User Pool to obtain inbound authorization JWT
    client = boto3.client('cognito-idp', region_name=REGION)
    try:
        auth_response = client.initiate_auth(
            ClientId=CLIENT_ID,
            AuthFlow='USER_PASSWORD_AUTH',
            AuthParameters={'USERNAME': USERNAME, 'PASSWORD': PASSWORD}
        )
        access_token = auth_response['AuthenticationResult']['AccessToken']
        print(f"✅ Cognito Authentication Successful. Bearer Token acquired: {access_token[:30]}...")
    except Exception as e:
        print(f"❌ Authentication Failed: {str(e)}")
        print("Check if the user exists and the Client ID is correct.")

### 2. Invoke the Agentcore Streaming Endpoint
This cell calls the live AWS Agentcore runtime. Responses are streamed in real-time.

In [ ]:
AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-s5aas5HZhy"
AGENTCORE_URL = f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{AGENT_ARN}/invocations"

def query_financial_agent(prompt: str):
    if access_token is None:
        print("❌ ERROR: Access token is missing. Please run Cell 1 (Cognito Authentication) successfully first.")
        return

    print(f"\n--- Query: {prompt} ---")
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": "demo-session-12345678901234567890123"
    }

    response = requests.post(
        AGENTCORE_URL, 
        headers=headers, 
        json={"prompt": prompt}, 
        stream=True
    )
    
    if response.status_code != 200:
        print(f"❌ Error {response.status_code}: {response.text}")
        return

    # Process the Server-Sent Events (SSE) stream
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode('utf-8')
            if decoded_line.startswith("data: "):
                data = json.loads(decoded_line[6:])
                print(data.get("event", ""), end="", flush=True)

### 3. Execute Required Financial Queries
Proving the agent's ability to handle real-time data and RAG retrieval.

In [ ]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]

for q in queries:
    query_financial_agent(q)

### 4. Langfuse Observability Trace Audit
Verify that the agent's reasoning steps were recorded.

In [ ]:
# Verifying traces using Langfuse API
langfuse_auth = ('pk-lf-...', 'sk-lf-...') # Replace with actual keys if required for live verification
trace_response = requests.get(
    "https://cloud.langfuse.com/api/public/traces?sessionId=demo-session-12345678901234567890123",
    auth=langfuse_auth
)

print("\n--- Langfuse Trace Data ---")
if trace_response.status_code == 200:
    print(json.dumps(trace_response.json(), indent=2))
else:
    print(f"❌ Could not fetch traces: {trace_response.status_code}")